# Rolling Risk Metrics

Use this notebook to develop rolling ETF fragility and risk diagnostics from the processed core panel.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

core_panel = pd.read_csv(config.CORE_PANEL_CSV, parse_dates=["Date", "Inception"])
core_panel = core_panel.sort_values(["Symbol", "Date"])
core_panel.head()

,Date,Symbol,Return,ANFCI,BAMLC0A0CM,DGS10,T10Y2Y,T5YIE,VIX,MOVE,...,Name,Assets,ETF Database Category,ER,Inception,RET_XS,age_years,Assets_clean,ER_clean,log_assets
0,2020-09-18,AAA,-0.002397,-0.683,1.35,0.70,0.56,1.58,25.830000,37.240002,...,Alternative Access First Priority CLO Bond ETF,"$42,609,200",Corporate Bonds,0.19%,2020-09-09,-0.002416,0.024641,42609200.0,0.0019,17.567581
1,2020-09-25,AAA,0.000200,-0.669,1.46,0.66,0.54,1.43,26.379999,36.970001,...,Alternative Access First Priority CLO Bond ETF,"$42,609,200",Corporate Bonds,0.19%,2020-09-09,0.000181,0.043806,42609200.0,0.0019,17.567581
2,2020-10-02,AAA,-0.002082,-0.631,1.43,0.70,0.57,1.48,27.629999,39.970001,...,Alternative Access First Priority CLO Bond ETF,"$42,609,200",Corporate Bonds,0.19%,2020-09-09,-0.002099,0.062971,42609200.0,0.0019,17.567581
3,2020-10-09,AAA,-0.000803,-0.580,1.35,0.79,0.63,1.56,25.000000,57.520000,...,Alternative Access First Priority CLO Bond ETF,"$42,609,200",Corporate Bonds,0.19%,2020-09-09,-0.000822,0.082136,42609200.0,0.0019,17.567581
4,2020-10-16,AAA,0.000603,-0.538,1.33,0.76,0.62,1.53,27.410000,57.250000,...,Alternative Access First Priority CLO Bond ETF,"$42,609,200",Corporate Bonds,0.19%,2020-09-09,0.000581,0.101300,42609200.0,0.0019,17.567581


In [2]:
window = 26

rolling = core_panel[["Symbol", "Date", "RET_XS"]].copy()
rolling["rolling_vol_26w"] = (
    rolling.groupby("Symbol")["RET_XS"]
    .rolling(window=window, min_periods=12)
    .std()
    .reset_index(level=0, drop=True)
)
rolling["rolling_var_10_26w"] = (
    rolling.groupby("Symbol")["RET_XS"]
    .rolling(window=window, min_periods=12)
    .quantile(0.10)
    .reset_index(level=0, drop=True)
)

rolling.tail()

,Symbol,Date,RET_XS,rolling_vol_26w,rolling_var_10_26w
156404,ZROZ,2026-03-06,-0.032876,0.018909,-0.021444
156405,ZROZ,2026-03-13,-0.035917,0.019197,-0.026589
156406,ZROZ,2026-03-20,-0.004929,0.018898,-0.026589
156407,ZROZ,2026-03-27,-0.006361,0.018917,-0.026589
156408,ZROZ,2026-04-03,0.022861,0.019394,-0.026589


In [3]:
latest = rolling.dropna().sort_values("Date").groupby("Symbol").tail(1)
latest.sort_values("rolling_vol_26w", ascending=False).head(20)

,Symbol,Date,RET_XS,rolling_vol_26w,rolling_var_10_26w
140078,TTT,2026-04-03,-0.048752,0.035782,-0.042569
139038,TMV,2026-04-03,-0.049961,0.035734,-0.040815
138518,TMF,2026-04-03,0.048715,0.035591,-0.049571
155888,XOVR,2026-04-03,0.028983,0.027485,-0.045926
132206,TBT,2026-04-03,-0.037342,0.023906,-0.027794
141638,UBT,2026-04-03,0.032512,0.023682,-0.032584
156408,ZROZ,2026-04-03,0.022861,0.019394,-0.026589
48787,GOVZ,2026-04-03,0.023027,0.019312,-0.027084
31341,FCVT,2026-04-03,0.035270,0.018666,-0.018695
22324,CWB,2026-04-03,0.039323,0.018144,-0.022449
